# WEEK 1 - Linear Regression: polynomial terms, interaction terms, multicollinearity, variance inflation factor and regression, and categorical and continuous features

In [ ]:
# !pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 53.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
import networkx as nx

In [5]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix

In [6]:
#1
df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")

#2
df_pima = pd.read_csv("pima_indian_diabetes_dataset.csv") 

Dataset 1

In [11]:
df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [12]:
df.columns

Index(['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [13]:
df['Diabetes_012'].value_counts()

Diabetes_012
0.0    213703
2.0     35346
1.0      4631
Name: count, dtype: int64

In [19]:
df['PhysActivity'].value_counts()

PhysActivity
1.0    191920
0.0     61760
Name: count, dtype: int64

In [7]:
selected_cols = ['BMI', 'PhysActivity', 'PhysHlth', 'Sex', 'Age', 'Education', 'Income']

df_BMIprediction = df[selected_cols]

df_BMIprediction.head(20)

,BMI,PhysActivity,PhysHlth,Sex,Age,Education,Income
0,40.0,0.0,15.0,0.0,9.0,4.0,3.0
1,25.0,1.0,0.0,0.0,7.0,6.0,1.0
2,28.0,0.0,30.0,0.0,9.0,4.0,8.0
3,27.0,1.0,0.0,0.0,11.0,3.0,6.0
4,24.0,1.0,0.0,0.0,11.0,5.0,4.0
5,25.0,1.0,2.0,1.0,10.0,6.0,8.0
6,30.0,0.0,14.0,0.0,9.0,6.0,7.0
7,25.0,1.0,0.0,0.0,11.0,4.0,4.0
8,30.0,0.0,30.0,0.0,9.0,5.0,1.0
9,24.0,0.0,0.0,1.0,8.0,4.0,3.0


In [27]:
# polynomial model

from sklearn.preprocessing import PolynomialFeatures

def polynomial_regression(df, target_col, degree=2):
    # Split predictors and target
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)
    feature_names = poly.get_feature_names_out(X.columns)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)
    
    # Fit model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    print(f"Polynomial degree: {degree}")
    print(f"R²: {r2:.3f}")
    print(f"MSE: {mse:.3f}")
    
    # Return coefficients if you want to inspect them
    return pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': model.coef_
    }).sort_values(by='Coefficient', key=abs, ascending=False)

In [33]:
coef_df = polynomial_regression(df_BMIprediction, target_col='BMI', degree=2)


Polynomial degree: 2
R²: 0.070
MSE: 40.441


In [34]:
coef_df = polynomial_regression(df, target_col='BMI', degree=2)

Polynomial degree: 2
R²: 0.178
MSE: 35.755


In [ ]:
#testing different degrees - Dataset 1

import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error

# --- Prepare data ---
X = df_BMIprediction.drop(columns=['BMI'])   # Replace with your target column
y = df_BMIprediction['BMI']

# --- Define k-fold cross-validation ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- Create storage for results ---
results = []

# --- Test degrees 1 through 5 ---
for degree in range(1, 6):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)

    model = LinearRegression()

    # Cross-validate R²
    r2_scores = cross_val_score(model, X_poly, y, cv=kf, scoring='r2')

    # Cross-validate MSE (note: use negative MSE because sklearn maximizes score)
    mse_scores = cross_val_score(
        model, X_poly, y, cv=kf,
        scoring=make_scorer(mean_squared_error, greater_is_better=False)
    )

    results.append({
        'Degree': degree,
        'Avg_R2': np.mean(r2_scores),
        'Avg_MSE': -np.mean(mse_scores)
    })

# --- Make results DataFrame ---
cv_results = pd.DataFrame(results)
print(cv_results)


   Degree    Avg_R2    Avg_MSE
0       1  0.042258  41.827685
1       2  0.070513  40.593954
2       3  0.072703  40.498276
3       4  0.074880  40.403179
4       5  0.074661  40.412782


There is a slight improvement between degree 1 and 2, but the benefit plateus above degree 2. Higher degree doesn't improve the performance and could start overfitting. 

Polynomial does not explain the data that well. 

Test for multicollinearity.

In [9]:
# Variance Inflation Factor (VIF) 

import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Separate predictors (drop the target column)
X = df_BMIprediction.drop(columns=['BMI'])

# Add a constant term for statsmodels
X_const = sm.add_constant(X)

# Calculate VIF for each feature
vif = pd.DataFrame()
vif["Feature"] = X_const.columns
vif["VIF"] = [variance_inflation_factor(X_const.values, i)
              for i in range(X_const.shape[1])]

# Display sorted VIFs
print(vif.sort_values(by="VIF", ascending=False))

        Feature        VIF
0         const  41.703061
6        Income   1.355580
5     Education   1.278046
2      PhysHlth   1.115069
1  PhysActivity   1.093442
4           Age   1.026661
3           Sex   1.018674


VIF all less than 2, so multico is not an issue.

Interaction Terms Dataset 1

In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error

# --- Prepare data ---
X = df_BMIprediction.drop(columns=['BMI'])
y = df_BMIprediction['BMI']

# --- Define k-fold cross-validation ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- Create storage for results ---
results = []

# --- Test polynomial degrees (only interactions) ---
for degree in range(1, 4):
    poly = PolynomialFeatures(degree=degree, include_bias=False, interaction_only=True)
    X_poly = poly.fit_transform(X)

    model = LinearRegression()

    # Cross-validate R²
    r2_scores = cross_val_score(model, X_poly, y, cv=kf, scoring='r2')

    # Cross-validate MSE (note: negative MSE because sklearn maximizes scores)
    mse_scores = cross_val_score(
        model, X_poly, y, cv=kf,
        scoring=make_scorer(mean_squared_error, greater_is_better=False)
    )

    results.append({
        'Degree': degree,
        'Avg_R2': np.mean(r2_scores),
        'Avg_MSE': -np.mean(mse_scores)
    })

# --- Make results DataFrame ---
cv_results_interactions = pd.DataFrame(results)
print(cv_results_interactions)


   Degree    Avg_R2    Avg_MSE
0       1  0.042258  41.827685
1       2  0.052443  41.382974
2       3  0.052842  41.365493


Interaction Terms - There is a 1% improvement in explained variance at degree 2. Some two variabel combos might have mild predictive value. Three way interactions at degree 3 do not not improve the model. 

Dataset 2

In [8]:
df_pima.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [15]:
df_pima['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [11]:
#predicting BMI since Outcome is categorical

df_pima_BMIprediction = df_pima.drop(columns=['Outcome'])
df_pima_BMIprediction.head(20)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
5,5,116,74,0,0,25.6,0.201,30
6,3,78,50,32,88,31.0,0.248,26
7,10,115,0,0,0,35.3,0.134,29
8,2,197,70,45,543,30.5,0.158,53
9,8,125,96,0,0,0.0,0.232,54


In [36]:
coef_pima_with_outcome = polynomial_regression(df_pima, target_col='BMI', degree=2)


Polynomial degree: 2
R²: -0.125
MSE: 79.995


In [37]:
coef_pima_no_outcome = polynomial_regression(df_pima_BMIprediction, target_col='BMI', degree=2)


Polynomial degree: 2
R²: -0.201
MSE: 85.352


Polynomial regression did not fit data well. 

In [ ]:
#testing different degrees

import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error

# --- Prepare data ---
X = df_pima_BMIprediction.drop(columns=['BMI'])   # Replace with your target column
y = df_pima_BMIprediction['BMI']

# --- Define k-fold cross-validation ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- Create storage for results ---
results = []

# --- Test degrees 1 through 5 ---
for degree in range(1, 6):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)

    model = LinearRegression()

    # Cross-validate R²
    r2_scores = cross_val_score(model, X_poly, y, cv=kf, scoring='r2')

    # Cross-validate MSE (note: use negative MSE because sklearn maximizes score)
    mse_scores = cross_val_score(
        model, X_poly, y, cv=kf,
        scoring=make_scorer(mean_squared_error, greater_is_better=False)
    )

    results.append({
        'Degree': degree,
        'Avg_R2': np.mean(r2_scores),
        'Avg_MSE': -np.mean(mse_scores)
    })

# --- Make results DataFrame ---
cv_results = pd.DataFrame(results)
print(cv_results)


   Degree        Avg_R2       Avg_MSE
0       1  2.064762e-01  4.873848e+01
1       2  1.863390e-01  5.099372e+01
2       3 -1.368168e-01  6.712685e+01
3       4 -1.842482e+02  1.271803e+04
4       5 -1.243289e+06  6.963954e+07


Added complexity does not improve the predictive power. Overfitting is already starting at degree 2. 

In [41]:
# Variance Inflation Factor (VIF) 

import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Separate predictors (drop the target column)
X = df_pima.drop(columns=['Outcome'])

# Add a constant term for statsmodels
X_const = sm.add_constant(X)

# Calculate VIF for each feature
vif = pd.DataFrame()
vif["Feature"] = X_const.columns
vif["VIF"] = [variance_inflation_factor(X_const.values, i)
              for i in range(X_const.shape[1])]

# Display sorted VIFs
print(vif.sort_values(by="VIF", ascending=False))


                    Feature        VIF
0                     const  35.039974
8                       Age   1.588368
4             SkinThickness   1.507432
1               Pregnancies   1.430872
5                   Insulin   1.427536
2                   Glucose   1.298961
6                       BMI   1.297450
3             BloodPressure   1.181863
7  DiabetesPedigreeFunction   1.067090


In [42]:
# Variance Inflation Factor (VIF) 

import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Separate predictors (drop the target column)
X = df_pima.drop(columns=['BMI'])

# Add a constant term for statsmodels
X_const = sm.add_constant(X)

# Calculate VIF for each feature
vif = pd.DataFrame()
vif["Feature"] = X_const.columns
vif["VIF"] = [variance_inflation_factor(X_const.values, i)
              for i in range(X_const.shape[1])]

# Display sorted VIFs
print(vif.sort_values(by="VIF", ascending=False))


                    Feature        VIF
0                     const  30.723808
7                       Age   1.592253
2                   Glucose   1.515232
1               Pregnancies   1.460358
5                   Insulin   1.428950
8                   Outcome   1.362977
4             SkinThickness   1.357542
3             BloodPressure   1.140101
6  DiabetesPedigreeFunction   1.081176


Multicollinearity is not an issue since VIF is less than 2. 

Interaction Terms

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error

# --- Prepare data ---
X = df_pima_BMIprediction.drop(columns=['BMI'])
y = df_pima_BMIprediction['BMI']

# --- Define k-fold cross-validation ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- Create storage for results ---
results = []

# --- Test polynomial degrees (only interactions) ---
for degree in range(1, 4):
    poly = PolynomialFeatures(degree=degree, include_bias=False, interaction_only=True)
    X_poly = poly.fit_transform(X)

    model = LinearRegression()

    # Cross-validate R²
    r2_scores = cross_val_score(model, X_poly, y, cv=kf, scoring='r2')

    # Cross-validate MSE (note: negative MSE because sklearn maximizes scores)
    mse_scores = cross_val_score(
        model, X_poly, y, cv=kf,
        scoring=make_scorer(mean_squared_error, greater_is_better=False)
    )

    results.append({
        'Degree': degree,
        'Avg_R2': np.mean(r2_scores),
        'Avg_MSE': -np.mean(mse_scores)
    })

# --- Make results DataFrame ---
cv_results_interactions = pd.DataFrame(results)
print(cv_results_interactions)


   Degree    Avg_R2    Avg_MSE
0       1  0.206476  48.738479
1       2  0.236969  46.906295
2       3  0.146347  51.631993


At degree 1, 20% of the variance is explained. Degree 2 adds pariwise interactions. There is about a 3% improvement in R2. This indicates the interactions do matter, though not a hughe improvement. At degree 3, the R2 and MSE are significantly worse, indicating the model is too complex and there is overfitting. 

--> add a resource describing what each degree is doing